# LunarLander-v3 — compact trained agent

A small policy ensemble for ML-Arena challenge **43**, using the notebook's
`flex_v1` contract. Two trained 8 → 64 → 64 → 4 networks combine their action
logits with weights 0.7 and 0.3. Inference uses NumPy only; upload **`agent.py`** and **`policy.npz`**.
The weights file is 38,464 bytes. PyTorch and the simulator are not inference
dependencies.

The goal is an **official platform** mean reward >= 200; stop submitting once
that score is **> 220**. Local calm and windy scores are reported separately.
Run the notebook from its directory with the supplied weights beside it.


## 0. Setup

In [ ]:
!pip install -q "mlarena-sdk==4.6.0" "gymnasium[box2d]==1.3.0" numpy

import mlarena

API_TOKEN = "mlk_user_be8edd1568655ff0_bbfbb4ff5c006ef6b5f81d63b919a32c"   # <-- paste your token (Profile page -> API Keys)
CHALLENGE_ID = 43

client = mlarena.connect(api_key=API_TOKEN, base_url="https://ml-arena.com")

## 1. Define your agent

The next cell writes **`agent.py`**. Keep **`policy.npz`** in the same directory;
the zero-argument constructor loads its six arrays with `allow_pickle=False`.
There are two 64-unit tanh hidden layers per policy, followed by four action
logits. The 0.7 / 0.3 ensemble weights are folded into the exported output layers.
The agent selects the action with the largest valid combined logit. It is deterministic and uses only the eight observed state values.

The worker calls `Agent()`, `setup(observation_space, action_space)` with
dict-encoded spaces, then `choose_action(...)` each step. Spaces are decoded
using the runtime's `flexkit`; terminal calls return `None`.


In [ ]:
%%writefile agent.py
"""Compact LunarLander-v3 policy ensemble; NumPy inference only."""

from pathlib import Path

import numpy as np


class Agent:
    def __init__(self):
        with np.load(Path(__file__).with_name("policy.npz"), allow_pickle=False) as data:
            self.layers = [(data[f"w{i}"], data[f"b{i}"]) for i in range(3)]

    def setup(self, observation_space, action_space):
        from flexkit.spaces import decode_space

        self.observation_space = decode_space(observation_space)
        self.action_space = decode_space(action_space)
        return True

    def choose_action(self, observation, reward=0.0, terminated=False,
                      truncated=False, info=None, action_mask=None):
        if terminated or truncated:
            return None
        x = np.broadcast_to(np.asarray(observation, dtype=np.float32), (2, 8))
        for weight, bias in self.layers[:-1]:
            x = np.tanh(np.einsum("nij,nj->ni", weight, x) + bias)
        weight, bias = self.layers[-1]
        logits = (np.einsum("nij,nj->ni", weight, x) + bias).mean(axis=0)
        if action_mask is not None:
            logits = np.where(action_mask, logits, -np.inf)
        return int(np.argmax(logits))


## 2. Evaluate locally

Use **1,000 unseen seeds (9000–9999)** per condition in `LunarLander-v3`, with
its 1,000-step limit. Evaluate both calm conditions and wind enabled (wind power
15, turbulence 1.5). These seeds were not used for training or checkpoint
selection. Gymnasium 1.3.0 matches the platform's advertised runtime version.
The platform does not publish its environment arguments, so its official score
remains the stopping criterion.

Local setup uses actual Gymnasium spaces directly because `flexkit` is supplied
by the platform. The exported NumPy ensemble matched the PyTorch ensemble's
chosen action on **10,000 test observations**; terminal and action-mask checks
also passed.


In [1]:
import gymnasium as gym
import numpy as np
from agent import Agent

agent = Agent()
assert agent.choose_action([0.0] * 8, terminated=True) is None
assert agent.choose_action([0.0] * 8, truncated=True) is None
for wind in (False, True):
    scores = []
    env = gym.make("LunarLander-v3", enable_wind=wind)
    agent.observation_space, agent.action_space = env.observation_space, env.action_space
    try:
        for seed in range(9000, 10000):
            observation, info = env.reset(seed=seed)
            total, reward = 0.0, 0.0
            for _ in range(env.spec.max_episode_steps):
                action = agent.choose_action(observation, reward=reward, info=info)
                assert type(action) is int and env.action_space.contains(action)
                observation, reward, terminated, truncated, info = env.step(action)
                total += reward
                if terminated or truncated:
                    assert agent.choose_action(observation, terminated=terminated, truncated=truncated) is None
                    break
            scores.append(total)
    finally:
        env.close()
    print(f"Wind={wind}: 1000 episodes, mean={np.mean(scores):.2f}, std={np.std(scores):.2f}")
    print(f"Episodes scoring >= 200: {sum(score >= 200 for score in scores)}/1000")


Wind=False: 1000 episodes, mean=263.25, std=19.70
Episodes scoring >= 200: 988/1000
Wind=True: 1000 episodes, mean=227.26, std=71.30
Episodes scoring >= 200: 830/1000


## 3. Submit

Upload `agent.py` and `policy.npz` together using the standard Python runtime,
then wait for the platform's evaluation. SDK 4.6.0 uses `challenge_id` and
`submission_id`; the original SDK 0.3.0's competition routes return HTTP 404.


In [2]:
result = client.submit(
    challenge_id=CHALLENGE_ID,
    files=["agent.py", "policy.npz"],
    submission_name="LunarLander calibrated ensemble",
    runtime={"language": "python", "framework": "none"},
)
print("Submission ID:", result["submission_id"])
for line in client.tail_logs(CHALLENGE_ID, result["submission_id"], timeout_sec=900):
    print(line)


Submission ID: 9525
[deploy_queue] Deployment queued successfully
  queued: position=1/1 waiting=0s
  run: job_status=pending steps=None score=None outcome=incomplete
  queued: position=1/1 waiting=5s
  queued: position=1/1 waiting=11s
[deploy_run] Deployment simulations started
  run: job_status=running steps=None score=None outcome=incomplete
[active] Deployment completed successfully
  run: job_status=completed steps=19219 score=241.96822911771545 outcome=None


## 4. Leaderboard

In [3]:
# Retrieve this exact submission's leaderboard row.
board = client.leaderboard(CHALLENGE_ID, top=1, aggregate="submission",
                           around=result["submission_id"], window=0)
row = board["around"]["row"] if isinstance(board, dict) else board.attrs["around"]["row"]
print({key: row[key] for key in ("submission_id", "score", "n_episodes_total", "rank")})


{'submission_id': 9525, 'score': 241.96822911771545, 'n_episodes_total': 50, 'rank': 242}


## 5. Training and validation

The controller was preserved as a fallback while training candidates offline.
The ensemble contains two PPO actors (Stable-Baselines3 2.9.0, PyTorch 2.14.1,
seed 42), each with two 64-unit tanh hidden layers. Training used 16 environments,
75% windy and 25% calm, rewards scaled by 0.01, gamma 0.999, GAE lambda 0.98,
and 1,024-step rollouts. One actor was trained from scratch then refined with
PPO; the second was initialized by behavior cloning the controller before PPO
refinement. Ensemble coefficients were calibrated on separate seeds 8000–8499.
The final coefficients 0.7 / 0.3 are folded into the exported output layers.
The critic and training framework are unnecessary for inference.

Validation used 1,000 episodes per condition, with the seeds shown:

| Agent | Seeds | Calm mean | Windy mean |
|---|---|---:|---:|
| Previous controller | 7000–7999 | 262.35 | 215.07 |
| Initial ensemble | 7000–7999 | 262.61 | 226.82 |
| Calibrated ensemble | 9000–9999 | 263.25 | 227.26 |

Official results so far: controller **9523** scored **199.55** over 35 episodes;
initial ensemble **9524** scored **219.46** over 50 episodes. The final official
result is recorded above. Local results and platform results are separate
evaluations. Development stops when the official score exceeds 220.

**Final official result:** submission **9525**, mean reward
**241.97** over **50 episodes**. This improves
on the previous controller's **199.55**. The official score exceeds **220**;
development and submissions stopped at this point, as requested.
